# Stage 1 — 도시 장애물 환경 학습 (로컬 GPU)

**담당**: 이재왕 (work/evader)  
**씬**: `Assets/01. Scenes/Stage1.unity`  
**목표**: 도시 환경에서 장애물 회피 + GoalZone 도달 학습 (Pursuer 없음)  
**전략**: Stage1-A (`_goalOnlyMode=true`, `_currentStage=1`)  
**수렴 기준**: `goal_reach_rate ≥ 30%`, `crash_rate ≤ 15%` → Stage1-B (RL Pursuer 추가)  

---

## 사전 준비 (최초 1회)

```bash
cd c:\IIT_DroneLearning
.venv\Scripts\activate
jupyter notebook python/notebooks/stage1_obstacle_local.ipynb
```

---

---
## 1. 환경 확인

In [39]:
import sys
import torch
import mlagents_envs

print(f'Python   : {sys.version.split()[0]}')
print(f'torch    : {torch.__version__}')
print(f'CUDA     : {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU      : {torch.cuda.get_device_name(0)}')
    print(f'VRAM     : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')
print(f'mlagents : {mlagents_envs.__version__}')


Python   : 3.10.11
torch    : 2.7.0+cu128
CUDA     : True
GPU      : NVIDIA GeForce RTX 5060
VRAM     : 8.5 GB
mlagents : 1.2.0.dev0


---
## 2. 경로 및 실험 설정

In [116]:
from pathlib import Path

REPO_PATH  = Path("c:/IIT_DroneLearning")
LOG_DIR    = REPO_PATH / "python" / "results"
CONFIG_DIR = REPO_PATH / "python" / "config"

LOG_DIR.mkdir(parents=True, exist_ok=True)

SEED = 42

# ===== Next-task profile =====
# "eval_v6"      : 학습 실행 없이 v6 결과 점검/평가
# "eval_v8"      : 학습 실행 없이 v8 결과 점검/평가
# "eval_v9"      : 학습 실행 없이 v9 결과 점검/평가
# "eval_v10"     : 학습 실행 없이 v10 결과 점검/평가
# "eval_v11"     : 학습 실행 없이 v11 결과 점검/평가
# "train_v7"     : v6를 초기값으로 v7 새 학습 시작 (--force + --initialize-from)
# "resume_v7"    : 기존 v7 이어학습 (--resume)
# "train_v8"     : v7를 초기값으로 v8(exp) 새 학습 시작 (--force + --initialize-from)
# "resume_v8"    : 기존 v8(exp) 이어학습 (--resume)
# "train_v9"     : v8를 초기값으로 v9(guard) 새 학습 시작 (--force + --initialize-from)
# "resume_v9"    : 기존 v9(guard) 이어학습 (--resume)
# "train_v10"    : v9를 초기값으로 v10(smooth) 새 학습 시작 (--force + --initialize-from)
# "resume_v10"   : 기존 v10(smooth) 이어학습 (--resume)
# "train_v11"    : v10을 초기값으로 v11(dewall) 새 학습 시작 (--force + --initialize-from)
# "resume_v11"   : 기존 v11(dewall) 이어학습 (--resume)
# "train_v12"    : v11(latest checkpoint, expected 249991) 기준 v12(wallfix) 새 학습 시작
# "resume_v12"   : 기존 v12(wallfix) 이어학습 (--resume)
# "train_v13"    : v12(latest checkpoint) 기준 v13(wallfocus) 새 학습 시작
# "resume_v13"   : 기존 v13(wallfocus) 이어학습 (--resume)
# "train_v13b"   : v13(latest checkpoint) 기준 v13b(boundaryguard) 새 학습 시작
# "resume_v13b"  : 기존 v13b(boundaryguard) 이어학습 (--resume)
# "train_v13c"   : v13b(latest checkpoint) 기준 v13c(rootfix) 새 학습 시작
# "resume_v13c"  : 기존 v13c(rootfix) 이어학습 (--resume)
WORKFLOW_MODE = "train_v13c"

# 중단 복구 시 목표 스텝까지 이어가려면 override를 사용하세요. (None이면 모드 기본값 사용)
TARGET_STOP_STEP_OVERRIDE = None

PREV_RUN_ID = f"evader_s1_obstacle_44d_v6_seed{SEED}"
V7_RUN_ID   = f"evader_s1_obstacle_44d_v7_seed{SEED}"
V8_RUN_ID   = f"evader_s1_obstacle_44d_v8exp_seed{SEED}"
V9_RUN_ID   = f"evader_s1_obstacle_44d_v9guard_seed{SEED}"
V10_RUN_ID  = f"evader_s1_obstacle_44d_v10smooth_seed{SEED}"
V11_RUN_ID  = f"evader_s1_obstacle_44d_v11dewall_seed{SEED}"
V12_RUN_ID  = f"evader_s1_obstacle_44d_v12wallfix_seed{SEED}"
V13_RUN_ID  = f"evader_s1_obstacle_44d_v13wallfocus_seed{SEED}"
V13B_RUN_ID = f"evader_s1_obstacle_44d_v13bguard_seed{SEED}"
V13C_RUN_ID = f"evader_s1_obstacle_44d_v13crootfix_seed{SEED}"

if WORKFLOW_MODE == "eval_v6":
    INIT_FROM = PREV_RUN_ID
    RUN_ID = PREV_RUN_ID
    RESUME = False
    TARGET_STOP_STEP = None
elif WORKFLOW_MODE == "eval_v8":
    INIT_FROM = ""
    RUN_ID = V8_RUN_ID
    RESUME = False
    TARGET_STOP_STEP = None
elif WORKFLOW_MODE == "eval_v9":
    INIT_FROM = ""
    RUN_ID = V9_RUN_ID
    RESUME = False
    TARGET_STOP_STEP = None
elif WORKFLOW_MODE == "eval_v10":
    INIT_FROM = ""
    RUN_ID = V10_RUN_ID
    RESUME = False
    TARGET_STOP_STEP = None
elif WORKFLOW_MODE == "eval_v11":
    INIT_FROM = ""
    RUN_ID = V11_RUN_ID
    RESUME = False
    TARGET_STOP_STEP = None
elif WORKFLOW_MODE == "train_v7":
    INIT_FROM = PREV_RUN_ID
    RUN_ID = V7_RUN_ID
    RESUME = False
    TARGET_STOP_STEP = 500000
elif WORKFLOW_MODE == "resume_v7":
    INIT_FROM = ""
    RUN_ID = V7_RUN_ID
    RESUME = True
    TARGET_STOP_STEP = 1000000
elif WORKFLOW_MODE == "train_v8":
    INIT_FROM = V7_RUN_ID
    RUN_ID = V8_RUN_ID
    RESUME = False
    TARGET_STOP_STEP = 300000
elif WORKFLOW_MODE == "resume_v8":
    INIT_FROM = ""
    RUN_ID = V8_RUN_ID
    RESUME = True
    TARGET_STOP_STEP = 600000
elif WORKFLOW_MODE == "train_v9":
    INIT_FROM = V8_RUN_ID
    RUN_ID = V9_RUN_ID
    RESUME = False
    TARGET_STOP_STEP = 300000
elif WORKFLOW_MODE == "resume_v9":
    INIT_FROM = ""
    RUN_ID = V9_RUN_ID
    RESUME = True
    TARGET_STOP_STEP = 600000
elif WORKFLOW_MODE == "train_v10":
    INIT_FROM = V9_RUN_ID
    RUN_ID = V10_RUN_ID
    RESUME = False
    TARGET_STOP_STEP = 300000
elif WORKFLOW_MODE == "resume_v10":
    INIT_FROM = ""
    RUN_ID = V10_RUN_ID
    RESUME = True
    TARGET_STOP_STEP = 600000
elif WORKFLOW_MODE == "train_v11":
    INIT_FROM = V10_RUN_ID
    RUN_ID = V11_RUN_ID
    RESUME = False
    TARGET_STOP_STEP = 250000
elif WORKFLOW_MODE == "resume_v11":
    INIT_FROM = ""
    RUN_ID = V11_RUN_ID
    RESUME = True
    TARGET_STOP_STEP = 500000
elif WORKFLOW_MODE == "train_v12":
    INIT_FROM = V11_RUN_ID
    RUN_ID = V12_RUN_ID
    RESUME = False
    TARGET_STOP_STEP = 100000
elif WORKFLOW_MODE == "resume_v12":
    INIT_FROM = ""
    RUN_ID = V12_RUN_ID
    RESUME = True
    TARGET_STOP_STEP = 250000
elif WORKFLOW_MODE == "train_v13":
    INIT_FROM = V12_RUN_ID
    RUN_ID = V13_RUN_ID
    RESUME = False
    TARGET_STOP_STEP = 100000
elif WORKFLOW_MODE == "resume_v13":
    INIT_FROM = ""
    RUN_ID = V13_RUN_ID
    RESUME = True
    TARGET_STOP_STEP = 250000
elif WORKFLOW_MODE == "train_v13b":
    INIT_FROM = V13_RUN_ID
    RUN_ID = V13B_RUN_ID
    RESUME = False
    TARGET_STOP_STEP = 100000
elif WORKFLOW_MODE == "resume_v13b":
    INIT_FROM = ""
    RUN_ID = V13B_RUN_ID
    RESUME = True
    TARGET_STOP_STEP = 250000
elif WORKFLOW_MODE == "train_v13c":
    INIT_FROM = V13B_RUN_ID
    RUN_ID = V13C_RUN_ID
    RESUME = False
    TARGET_STOP_STEP = 100000
elif WORKFLOW_MODE == "resume_v13c":
    INIT_FROM = ""
    RUN_ID = V13C_RUN_ID
    RESUME = True
    TARGET_STOP_STEP = 250000
else:
    raise ValueError(f"Unknown WORKFLOW_MODE: {WORKFLOW_MODE}")

if WORKFLOW_MODE in {"train_v13c", "resume_v13c"}:
    CONFIG_NAME = "evader_s1_obstacle_20260408_v13c_rootfix.yaml"
elif WORKFLOW_MODE in {"train_v13b", "resume_v13b"}:
    CONFIG_NAME = "evader_s1_obstacle_20260408_v13b_boundaryguard.yaml"
elif WORKFLOW_MODE in {"train_v13", "resume_v13"}:
    CONFIG_NAME = "evader_s1_obstacle_20260408_v13_wallfocus.yaml"
elif WORKFLOW_MODE in {"train_v12", "resume_v12"}:
    CONFIG_NAME = "evader_s1_obstacle_20260409_v12_wallfix.yaml"
elif WORKFLOW_MODE in {"train_v11", "resume_v11", "eval_v11"}:
    CONFIG_NAME = "evader_s1_obstacle_20260408_v11_dewall.yaml"
else:
    CONFIG_NAME = "evader_s1_obstacle_template.yaml"

if WORKFLOW_MODE not in {"eval_v6", "eval_v8", "eval_v9", "eval_v10", "eval_v11"} and TARGET_STOP_STEP_OVERRIDE is not None:
    TARGET_STOP_STEP = TARGET_STOP_STEP_OVERRIDE

if WORKFLOW_MODE == "eval_v6":
    EVAL_RUN_ID = PREV_RUN_ID
elif WORKFLOW_MODE == "eval_v8":
    EVAL_RUN_ID = V8_RUN_ID
elif WORKFLOW_MODE == "eval_v9":
    EVAL_RUN_ID = V9_RUN_ID
elif WORKFLOW_MODE == "eval_v10":
    EVAL_RUN_ID = V10_RUN_ID
elif WORKFLOW_MODE == "eval_v11":
    EVAL_RUN_ID = V11_RUN_ID
else:
    EVAL_RUN_ID = RUN_ID

EVAL_EPISODES = 50
EVAL_SEED = 42

print(f"Repo          : {REPO_PATH}")
print(f"Log dir       : {LOG_DIR}")
print(f"Workflow mode : {WORKFLOW_MODE}")
print(f"Run ID        : {RUN_ID}")
print(f"Config file   : {CONFIG_NAME}")
print(f"Init From     : {INIT_FROM or '(none)'}")
if WORKFLOW_MODE == "train_v12":
    print("Init checkpoint hint: uses latest checkpoint in v11 run (expected step 249991).")
if WORKFLOW_MODE == "train_v13":
    print("Init checkpoint hint: uses latest checkpoint in v12 run.")
if WORKFLOW_MODE == "train_v13b":
    print("Init checkpoint hint: uses latest checkpoint in v13 run.")
if WORKFLOW_MODE == "train_v13c":
    print("Init checkpoint hint: uses latest checkpoint in v13b run.")
print(f"Resume        : {RESUME}")
print(f"Target step   : {TARGET_STOP_STEP}")
print(f"Target ovrd   : {TARGET_STOP_STEP_OVERRIDE}")
print(f"Eval run-id   : {EVAL_RUN_ID} / episodes={EVAL_EPISODES} / seed={EVAL_SEED}")


Repo          : c:\IIT_DroneLearning
Log dir       : c:\IIT_DroneLearning\python\results
Workflow mode : train_v13c
Run ID        : evader_s1_obstacle_44d_v13crootfix_seed42
Config file   : evader_s1_obstacle_20260408_v13c_rootfix.yaml
Init From     : evader_s1_obstacle_44d_v13bguard_seed42
Init checkpoint hint: uses latest checkpoint in v13b run.
Resume        : False
Target step   : 100000
Target ovrd   : None
Eval run-id   : evader_s1_obstacle_44d_v13crootfix_seed42 / episodes=50 / seed=42


### 2-1. 추가 학습 빠른 세팅 (권장)

아래 코드 셀을 실행하면 현재 베이스라인(`v13crootfix`)에서 **이어학습(resume)** 으로 바로 맞춰집니다.
기본 타깃 스텝은 `250000`으로 설정되어 있습니다.

In [120]:
# 추가 학습 preset: v13c run 이어학습
WORKFLOW_MODE = "resume_v13c"
TARGET_STOP_STEP_OVERRIDE = 250000

if WORKFLOW_MODE == "resume_v13c":
    INIT_FROM = ""
    RUN_ID = V13C_RUN_ID
    RESUME = True
    TARGET_STOP_STEP = TARGET_STOP_STEP_OVERRIDE if TARGET_STOP_STEP_OVERRIDE is not None else 250000
    CONFIG_NAME = "evader_s1_obstacle_20260408_v13c_rootfix.yaml"
    EVAL_RUN_ID = RUN_ID
else:
    raise ValueError(f"이 셀은 resume_v13c 전용입니다. 현재: {WORKFLOW_MODE}")

config_path = CONFIG_DIR / CONFIG_NAME
assert config_path.exists(), f"Config 없음: {config_path}"

print("[override] 추가 학습 세팅 적용 완료")
print(f"  WORKFLOW_MODE: {WORKFLOW_MODE}")
print(f"  RUN_ID       : {RUN_ID}")
print(f"  RESUME       : {RESUME}")
print(f"  TARGET_STEP  : {TARGET_STOP_STEP}")
print(f"  CONFIG       : {CONFIG_NAME}")

[override] 추가 학습 세팅 적용 완료
  WORKFLOW_MODE: resume_v13c
  RUN_ID       : evader_s1_obstacle_44d_v13crootfix_seed42
  RESUME       : True
  TARGET_STEP  : 250000
  CONFIG       : evader_s1_obstacle_20260408_v13c_rootfix.yaml


---
## 3. Config 확인

In [121]:
config_path = CONFIG_DIR / CONFIG_NAME
assert config_path.exists(), f'Config 없음: {config_path}'

print(f'Config: {config_path}{"=" * 60}')
print(config_path.read_text(encoding='utf-8'))


Config: c:\IIT_DroneLearning\python\config\evader_s1_obstacle_20260408_v13c_rootfix.yaml============================================================
# evader_s1_obstacle_20260408_v13c_rootfix.yaml
# Stage1-A v13c: root-cause collision hardening from v13b warm-start.
#
# Intended run:
#   --run-id=evader_s1_obstacle_44d_v13crootfix_seed42
#   --initialize-from=evader_s1_obstacle_44d_v13bguard_seed42
#
# Notes:
# - Keep model architecture unchanged (44D vector + ray)
# - Use conservative PPO update settings to stabilize after runtime safety/reward changes

engine_settings:
  time_scale: 20
  capture_frame_rate: 60

behaviors:
  Drone_Evader:
    trainer_type: ppo

    hyperparameters:
      batch_size: 256
      buffer_size: 28000
      learning_rate: 1.15e-4
      beta: 2.6e-3
      epsilon: 0.12
      lambd: 0.99
      num_epoch: 3
      learning_rate_schedule: linear

    network_settings:
      normalize: true
      hidden_units: 256
      num_layers: 2

    reward_signals:
      ext

---
## 4. Unity Editor 연결 확인

⚠️ Unity에서 `Stage1.unity` 씬을 열고 Inspector 값을 먼저 확인하세요.  
`WORKFLOW_MODE = "eval_v6"`, `"eval_v8"`, `"eval_v9"`, `"eval_v10"`, `"eval_v11"` 일 때는 5번 학습 셀을 실행하지 않습니다.  
학습을 재시작할 때만 `WORKFLOW_MODE = "train_v7"`, `"resume_v7"`, `"train_v8"`, `"resume_v8"`, `"train_v9"`, `"resume_v9"`, `"train_v10"`, `"resume_v10"`, `"train_v11"`, `"resume_v11"`, `"train_v12"`, `"resume_v12"`, `"train_v13"`, `"resume_v13"`, `"train_v13b"`, `"resume_v13b"`, `"train_v13c"`, `"resume_v13c"`로 변경하세요.

**Unity Inspector 체크리스트 (v13c rootfix + 추가학습 기준):**

| 컴포넌트 | 필드 | 값 |
|---|---|---|
| **Goal** | **Layer** | **Ignore Raycast** (필수) |
| EpisodeSpawnCoordinator | Strategy | SpawnCenterRandom |
| SpawnCenter | Radius | 12 |
| EvaderAgent | _currentStage | 1 |
| EvaderAgent | _goalOnlyMode | true |
| EvaderAgent | Max Episode Seconds | 40 |
| EvaderAgent | _goalArrivalReward | 3.0 |
| EvaderAgent | _checkpointCount | 2 |
| EvaderAgent | _checkpointRadius | 4.0 |
| EvaderAgent | _checkpointReward | 0.10 |
| EvaderAgent | _autoSetGoalIgnoreRaycastLayer | true |
| EvaderAgent | _excludeIgnoreRaycastFromSensorMask | true |
| EvaderAgent | _forceNonEmptyDetectionLayerMask | true |
| EvaderAgent | _disableMiddleBottomRaysInStage1 | true |
| EvaderAgent | _disableBottomRayInStage1 | true |
| EvaderAgent | _validateGoalPlacement | true |
| EvaderAgent | _goalObstacleClearanceRadius | 14.0 (v13c 권장) |
| EvaderAgent | _goalResampleMaxAttempts | 120 (v13c 권장) |
| EvaderAgent | _goalBoundaryClearanceMargin | 6.0 (v13c 권장) |
| EvaderAgent | _useStage1RewardPreset | true |
| EvaderAgent | _stage1RewardPreset | Experimental (롤백: LegacyV7) |
| EvaderAgent (Experimental) | VelAlignCoeff | 0.013 |
| EvaderAgent (Experimental) | GoalPriorityDist | 7.0 |
| EvaderAgent (Experimental) | ProximityCoeff | 0.0060 |
| EvaderAgent (Experimental) | ProximityThreshold | 0.24 |
| EvaderAgent (Experimental) | NearGoalObstaclePenaltyScale | 0.78 |
| EvaderAgent (Experimental) | GoalApproachObstaclePenaltyScale | 0.70 |
| EvaderAgent (Experimental) | TimePenaltyPerStep | -0.0015 |
| EvaderAgent (Experimental) | SurvivalRewardPerStep | 0.0008 |
| EvaderAgent (Experimental) | NearGoalShapingMultiplier | 1.8 |
| EvaderAgent (Experimental) | MiddleBottomRayPenaltyWeight | 0.12 |
| EvaderAgent (Experimental) | BottomRayPenaltyWeight | 0.0 |
| EvaderAgent | _useActionSmoothing | true |
| EvaderAgent | _actionDeadzone | 0.03 |
| EvaderAgent | _commandLerpFactor | 0.24 (추가학습 권장) |
| EvaderAgent | _maxCommandDeltaPerStep | 0.14 (추가학습 권장) |
| EvaderAgent | _enableWallProximitySafety | true |
| EvaderAgent | _wallProximityThreshold | 0.14 (v13c 권장) |
| EvaderAgent | _blockForwardPitchOnWallProximity | true |
| EvaderAgent | _wallForwardBlockEnterRisk | 0.52 (추가학습 권장) |
| EvaderAgent | _wallForwardBlockReleaseRisk | 0.24 (추가학습 권장) |
| EvaderAgent | _wallReverseAssistStartRisk | 0.40 (추가학습 권장) |
| EvaderAgent | _wallReverseAssistMaxPitch | 0.16 (추가학습 권장) |
| EvaderAgent | _wallAvoidRollAssist | 0.18 (v13c 권장) |
| EvaderAgent | _wallAvoidYawAssist | 0.14 (v13c 권장) |
| EvaderAgent | _wallAvoidBiasSmoothing | 0.18 (추가학습 권장) |
| EvaderAgent | _wallSymmetricClearanceBiasScale | 0.10 (추가학습 권장) |
| EvaderAgent | _enableBoundarySafety | true |
| EvaderAgent | _boundarySafetyMargin | 12.0 (v13c 권장) |
| EvaderAgent | _boundaryInwardRollAssist | 0.20 (v13c 권장) |
| EvaderAgent | _boundaryInwardYawAssist | 0.26 (v13c 권장) |
| EvaderAgent | _boundaryForwardPitchMinScale | 0.15 (v13c 권장) |
| EvaderAgent | _boundaryRiskPenaltyPerStep | -0.0030 (v13c 권장) |
| EvaderAgent | _boundaryOutwardSpeedReference | 6.0 (v13c 권장) |
| EvaderAgent | _boundaryEmergencyRiskThreshold | 0.88 (v13c 권장) |
| EvaderAgent | _spawnBoundaryInset | 2.0 (v13c 권장) |
| EvaderAgent | _terminalPenaltyBoundaryOverflow | -1.25 |
| EvaderAgent | _terminalPenaltyOuterWall | -1.10 |
| EvaderAgent | _terminalPenaltyBuilding | -1.60 |
| EvaderAgent | _terminalPenaltyOtherCrash | -1.00 |
| EvaderAgent | _enableStagnationPenalty | true |
| EvaderAgent | _stagnationWatchDistance | 18 |
| EvaderAgent | _stagnationProgressThreshold | 0.010 |
| EvaderAgent | _stagnationGraceSteps | 40 |
| EvaderAgent | _stagnationPenaltyPerStep | -0.0012 |
| EvaderReward | _goalShapingCoeff | 0.28 |
| EvaderReward | _farGoalShapingBoostDistance | 18 |
| EvaderReward | _farGoalShapingBoostMultiplier | 1.6 |
| EvaderReward | _goalApproachObstaclePenaltyScale | 0.70 |
| EvaderReward | _timePenaltyPerStep | -0.0015 |
| EvaderReward | _survivalRewardPerStep | 0.0008 |
| BehaviorParameters | Behavior Name | Drone_Evader |
| BehaviorParameters | Behavior Type | Default |

In [118]:
import socket

port = 5004
with socket.socket(socket.AF_INET, socket.SOCK_STREAM) as s:
    result = s.connect_ex(('127.0.0.1', port))
    if result == 0:
        print(f'⚠️  포트 {port} 이미 사용 중 — 이전 mlagents-learn 프로세스가 남아있을 수 있습니다.')
    else:
        print(f'✅ 포트 {port} 사용 가능. 다음 셀 실행 후 Unity에서 Play 하세요.')


✅ 포트 5004 사용 가능. 다음 셀 실행 후 Unity에서 Play 하세요.


---
## 5. 학습 실행 (train/resume 모드 전용)

이 셀은 `WORKFLOW_MODE`가 `train_v7`, `resume_v7`, `train_v8`, `resume_v8`, `train_v9`, `resume_v9`, `train_v10`, `resume_v10`, `train_v11`, `resume_v11`, `train_v12`, `resume_v12`, `train_v13`, `resume_v13`, `train_v13b`, `resume_v13b`, `train_v13c`, `resume_v13c`일 때만 실행하세요.  
`eval_v6`, `eval_v8`, `eval_v9`, `eval_v10`, `eval_v11` 모드에서는 셀이 자동으로 skip 됩니다.

중간 중단 복구 규칙:
- 코드 수정/Unity 재컴파일/수동 중단 후에는 같은 `RUN_ID`로 `resume_*` 모드를 사용합니다.
- 이미 목표 step 이상이면 셀이 자동으로 skip 됩니다.

실행 순서:
1. 이 셀 실행 (mlagents-learn 대기)
2. Unity Editor -> Stage1.unity -> Play
3. target step 도달 시 자동 중단

In [119]:
import subprocess, sys, socket, time, threading, json, os, re
from pathlib import Path

mode = globals().get("WORKFLOW_MODE", "train_v13c")
EVAL_MODES = {"eval_v6", "eval_v8", "eval_v9", "eval_v10", "eval_v11"}
if mode in EVAL_MODES:
    print(f"[skip] WORKFLOW_MODE='{mode}' 입니다. 학습 셀을 건너뜁니다.")
    print("      학습을 하려면 설정 셀에서 train/resume 모드로 바꾼 뒤 다시 실행하세요.")
    raise SystemExit(0)

venv_scripts = Path(sys.executable).parent
mlagents_bin = venv_scripts / 'mlagents-learn.exe'
BASE_PORT = 5004
RETRY_ON_PORT_IN_USE = True

# RESUME=True: --resume (이어받기) / False: --force + warm-start
cmd = [
    str(mlagents_bin), str(config_path),
    f'--run-id={RUN_ID}',
    f'--results-dir={LOG_DIR}',
]
if RESUME:
    cmd += ['--resume']
    print('Mode: RESUME — 마지막 체크포인트에서 이어받습니다.')
else:
    cmd += ['--force']
    if INIT_FROM:
        cmd += [f'--initialize-from={INIT_FROM}']
        print(f'Mode: FORCE — Warm-start from: {INIT_FROM}')
    else:
        print('Mode: FORCE — 처음부터 학습합니다.')

# 목표 스텝 도달 시 안전 중단 (None이면 비활성)
if 'TARGET_STOP_STEP' not in globals():
    TARGET_STOP_STEP = 500000
POLL_INTERVAL_SEC = 2
MONITOR_HEARTBEAT_SEC = 30
MONITOR_STALE_WARN_SEC = 120
FRESH_SOURCE_SEC = 90

print('실행 커맨드:')
print(' '.join(str(c) for c in cmd))
print()
if TARGET_STOP_STEP is not None:
    print(f'Auto-stop target step: {TARGET_STOP_STEP}')

run_dir = LOG_DIR / RUN_ID
run_dir.mkdir(parents=True, exist_ok=True)
init_metadata_path = run_dir / 'init_metadata.json'
init_metadata = {
    'workflow_mode': WORKFLOW_MODE,
    'run_id': RUN_ID,
    'config': CONFIG_NAME,
    'resume': RESUME,
    'initialize_from': INIT_FROM,
    'target_stop_step': TARGET_STOP_STEP,
    'seed': SEED,
    'created_utc': time.strftime('%Y-%m-%dT%H:%M:%SZ', time.gmtime()),
}
init_metadata_path.write_text(json.dumps(init_metadata, ensure_ascii=True, indent=2), encoding='utf-8')
print(f'[init] metadata saved: {init_metadata_path}')

output_lines = []

state_lock = threading.Lock()
step_state = {
    'stream_step': None,
    'stream_updated_at': None,
    'file_step': None,
    'file_updated_at': None,
    'file_source': None,
}
runtime_state = {
    'stop_requested': False,
    'suppressed_shutdown_lines': 0,
}
NOISY_SHUTDOWN_MARKERS = [
    '[ERROR] UnityEnvironment worker 0: environment raised an unexpected exception.',
    'subprocess_env_manager.py',
    'BrokenPipeError',
    'EOFError',
    'forrtl: error (200): program aborting due to control-BREAK event',
]

# Example lines:
# [INFO] Drone_Evader. Step: 360000. Time Elapsed: ...
# [INFO] Resuming training from step 353848.
step_pattern = re.compile(r"Step:\s*(\d+)")
resume_pattern = re.compile(r"Resuming training from step\s*(\d+)")
onnx_pattern = re.compile(r"-(\d+)\.onnx$")
model_step_pattern = re.compile(r"-(\d+)\.(onnx|pt)$")


def kill_stale_mlagents_processes():
    if os.name != 'nt':
        return
    try:
        subprocess.run(
            ['taskkill', '/F', '/IM', 'mlagents-learn.exe', '/T'],
            capture_output=True,
            text=True,
            check=False,
        )
    except Exception as e:
        print(f'[warn] stale process cleanup failed: {e}')


def port_is_busy(port_number: int) -> bool:
    with socket.socket(socket.AF_INET, socket.SOCK_STREAM) as s:
        return s.connect_ex(('127.0.0.1', port_number)) == 0


def update_stream_step(step_value: int):
    now_ts = time.time()
    with state_lock:
        prev = step_state['stream_step']
        if prev is None:
            step_state['stream_step'] = step_value
        else:
            # 출력 지연/역순 로그가 섞여도 step은 후퇴하지 않게 유지
            step_state['stream_step'] = max(prev, step_value)
        step_state['stream_updated_at'] = now_ts


def stream_output(pipe):
    for line in pipe:
        text = line.decode('utf-8', errors='replace').rstrip()
        output_lines.append(text)

        m_step = step_pattern.search(text)
        if m_step:
            try:
                update_stream_step(int(m_step.group(1)))
            except ValueError:
                pass

        m_resume = resume_pattern.search(text)
        if m_resume:
            try:
                update_stream_step(int(m_resume.group(1)))
            except ValueError:
                pass

        # Exported ... Drone_Evader-349884.onnx 형태도 스텝으로 반영
        if 'Exported ' in text and '.onnx' in text:
            try:
                tail = text.split()[-1]
                m_onnx = onnx_pattern.search(tail)
                if m_onnx:
                    update_stream_step(int(m_onnx.group(1)))
            except Exception:
                pass

        with state_lock:
            stop_phase = runtime_state['stop_requested']

        if stop_phase and any(marker in text for marker in NOISY_SHUTDOWN_MARKERS):
            with state_lock:
                runtime_state['suppressed_shutdown_lines'] += 1
            continue

        print(text)


def read_step_snapshot(log_root: Path, run_id: str):
    """run_logs + model artifacts에서 가능한 최신 step 후보를 읽고 (step, mtime, source) 반환."""
    run_root = log_root / run_id
    run_logs_dir = run_root / 'run_logs'
    model_dir = run_root / 'Drone_Evader'
    timers_path = run_logs_dir / 'timers.json'
    status_path = run_logs_dir / 'training_status.json'

    candidates = []

    if timers_path.exists():
        try:
            timers_mtime = timers_path.stat().st_mtime
            timers = json.loads(timers_path.read_text(encoding='utf-8'))
            gauges = timers.get('gauges', {})
            step_keys = [k for k in gauges.keys() if k.endswith('.Step.mean')]
            if step_keys:
                val = gauges[step_keys[0]].get('value', None)
                if isinstance(val, (int, float)):
                    candidates.append((int(val), timers_mtime, 'timers'))
        except Exception:
            pass

    if status_path.exists():
        try:
            status_mtime = status_path.stat().st_mtime
            status = json.loads(status_path.read_text(encoding='utf-8'))
            for key, val in status.items():
                if key == 'metadata' or not isinstance(val, dict):
                    continue
                final_ckpt = val.get('final_checkpoint', {})
                step = final_ckpt.get('steps', None)
                if isinstance(step, int):
                    candidates.append((step, status_mtime, 'status'))
        except Exception:
            pass

    if model_dir.exists():
        try:
            for model_file in model_dir.glob('Drone_Evader-*.*'):
                if not model_file.is_file():
                    continue
                m_model = model_step_pattern.search(model_file.name)
                if not m_model:
                    continue
                model_step = int(m_model.group(1))
                model_kind = m_model.group(2)
                candidates.append((model_step, model_file.stat().st_mtime, f'artifact_{model_kind}'))
        except Exception:
            pass

    if not candidates:
        return None, None, None

    best_step = max(c[0] for c in candidates)
    best_mtime = max(c[1] for c in candidates if c[0] == best_step)
    best_sources = sorted({c[2] for c in candidates if c[0] == best_step})
    return best_step, best_mtime, '+'.join(best_sources)


# 실행 전 pre-check: 이미 target 이상이면 프로세스를 시작하지 않는다.
pre_step, pre_step_mtime, pre_step_source = read_step_snapshot(LOG_DIR, RUN_ID)
if TARGET_STOP_STEP is not None and pre_step is not None:
    age_txt = 'unknown'
    if pre_step_mtime is not None:
        age_txt = f'{(time.time() - pre_step_mtime):.0f}s'
    print(f'[pre-check] snapshot_step={pre_step} / target={TARGET_STOP_STEP} / source={pre_step_source or "n/a"} / age={age_txt}')

    if RESUME and pre_step >= TARGET_STOP_STEP:
        print()
        print('학습 시작 스킵: 현재 체크포인트가 이미 target step 이상입니다.')
        print(f'  - current_step={pre_step}, target={TARGET_STOP_STEP}')
        print('  - 더 학습하려면 TARGET_STOP_STEP을 더 크게 설정하세요. (예: 600000)')
        raise SystemExit(0)

    if not RESUME:
        print('[pre-check] FORCE 모드에서는 기존 snapshot step을 auto-skip 기준으로 사용하지 않습니다.')

if port_is_busy(BASE_PORT):
    print(f'[pre-check] port {BASE_PORT} is busy -> stale mlagents-learn cleanup 시도')
    kill_stale_mlagents_processes()
    time.sleep(2)
    if port_is_busy(BASE_PORT):
        print(f'[pre-check] port {BASE_PORT} still busy after cleanup. Unity Play 또는 이전 세션을 확인하세요.')

print(f'⏳ mlagents-learn 시작 중... (포트 {BASE_PORT} 준비 대기)')

# CTRL_BREAK_EVENT는 일부 런타임(forrtl)에서 noisy abort를 유발할 수 있어 사용하지 않음.
creationflags = 0

attempt = 0
max_attempts = 2
while attempt < max_attempts:
    if attempt > 0:
        print(f'[retry] mlagents-learn 재시도 {attempt + 1}/{max_attempts}')

    output_lines.clear()
    with state_lock:
        step_state['stream_step'] = None
        step_state['stream_updated_at'] = None
        step_state['file_step'] = None
        step_state['file_updated_at'] = None
        step_state['file_source'] = None
        runtime_state['stop_requested'] = False
        runtime_state['suppressed_shutdown_lines'] = 0

    attempt_started_at = time.time()

    proc = subprocess.Popen(
        cmd, cwd=str(REPO_PATH),
        stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
        creationflags=creationflags
    )
    t = threading.Thread(target=stream_output, args=(proc.stdout,), daemon=True)
    t.start()

    port = BASE_PORT
    ready = False
    for _ in range(60):
        time.sleep(0.5)
        if proc.poll() is not None:
            t.join(timeout=2)
            break
        with socket.socket(socket.AF_INET, socket.SOCK_STREAM) as s:
            s.settimeout(0.3)
            if s.connect_ex(('127.0.0.1', port)) == 0:
                ready = True
                break

    if not ready and proc.poll() is not None:
        captured_output = '\n'.join(output_lines)
        if RETRY_ON_PORT_IN_USE and attempt == 0 and ('UnityWorkerInUseException' in captured_output or 'Failed to bind to address' in captured_output):
            print('[retry] worker/port in use detected. stale process cleanup 후 재시도합니다.')
            kill_stale_mlagents_processes()
            time.sleep(2)
            attempt += 1
            continue
        print(f'\n❌ mlagents-learn이 즉시 종료되었습니다 (exit={proc.returncode})')
        print(captured_output)
        raise RuntimeError('mlagents-learn 시작 실패 — 위 출력 내용을 확인하세요.')

    if ready:
        print()
        print('=' * 55)
        print(f'✅ mlagents-learn 준비 완료 — 포트 {BASE_PORT} listening')
        print('▶  지금 Unity Editor에서 Stage1.unity → Play 누르세요!')
        print('=' * 55)

        stop_requested = False
        last_monitor_step = None
        last_step_change_ts = time.time()
        last_heartbeat_print_ts = 0.0
        last_stale_warn_ts = 0.0

        while proc.poll() is None:
            time.sleep(POLL_INTERVAL_SEC)
            if TARGET_STOP_STEP is None:
                continue

            now_ts = time.time()

            file_step, file_mtime, file_source = read_step_snapshot(LOG_DIR, RUN_ID)
            if (
                file_step is not None
                and file_mtime is not None
                and file_mtime >= (attempt_started_at - 1)
):
                with state_lock:
                    prev_file_ts = step_state['file_updated_at']
                    prev_file_step = step_state['file_step']
                    if prev_file_ts is None or file_mtime >= prev_file_ts:
                        if prev_file_step is None:
                            step_state['file_step'] = file_step
                        else:
                            step_state['file_step'] = max(prev_file_step, file_step)
                        step_state['file_updated_at'] = file_mtime
                        step_state['file_source'] = file_source or 'file'

            with state_lock:
                stream_step = step_state['stream_step']
                stream_ts = step_state['stream_updated_at']
                file_step_cached = step_state['file_step']
                file_ts = step_state['file_updated_at']
                file_source_cached = step_state['file_source']

            candidates = []
            if stream_step is not None and stream_ts is not None:
                candidates.append(('stream', stream_step, now_ts - stream_ts))
            if file_step_cached is not None and file_ts is not None and file_ts >= (attempt_started_at - 1):
                candidates.append((file_source_cached or 'file', file_step_cached, now_ts - file_ts))

            if not candidates:
                if now_ts - last_heartbeat_print_ts >= MONITOR_HEARTBEAT_SEC:
                    last_heartbeat_print_ts = now_ts
                    print(f'[monitor:heartbeat][src:none] current_step=(none) / target={TARGET_STOP_STEP}')
                continue

            fresh_candidates = [c for c in candidates if c[2] <= FRESH_SOURCE_SEC]
            if fresh_candidates:
                source, current_step, source_age = max(fresh_candidates, key=lambda x: x[1])
            else:
                # 모두 오래된 값이면 가장 최근에 갱신된 소스를 사용
                source, current_step, source_age = min(candidates, key=lambda x: x[2])

            if source_age <= 5:
                age_tag = 'fresh'
            elif source_age <= FRESH_SOURCE_SEC:
                age_tag = f'age={source_age:.0f}s'
            else:
                age_tag = f'stale={source_age:.0f}s'

            changed = last_monitor_step is None or current_step != last_monitor_step
            if changed:
                last_monitor_step = current_step
                last_heartbeat_print_ts = now_ts
                # source가 충분히 신선할 때만 stale 타이머 리셋
                if source_age <= FRESH_SOURCE_SEC:
                    last_step_change_ts = now_ts
                print(f'[monitor:step-change][src:{source}][{age_tag}] current_step={current_step} / target={TARGET_STOP_STEP}')
            elif now_ts - last_heartbeat_print_ts >= MONITOR_HEARTBEAT_SEC:
                last_heartbeat_print_ts = now_ts
                print(f'[monitor:heartbeat][src:{source}][{age_tag}] current_step={current_step} / target={TARGET_STOP_STEP}')

            if (
                now_ts - last_step_change_ts >= MONITOR_STALE_WARN_SEC
                and now_ts - last_stale_warn_ts >= MONITOR_HEARTBEAT_SEC
):
                last_stale_warn_ts = now_ts
                if source_age > FRESH_SOURCE_SEC:
                    print('[monitor:stale-source] step 소스가 오래되었습니다. Unity Play 상태/로그 파일 갱신 여부를 확인하세요.')
                else:
                    print('[monitor:stale-progress] source는 응답 중이지만 step 증가가 없습니다. 환경 정체 여부를 확인하세요.')

            if current_step >= TARGET_STOP_STEP and not stop_requested:
                print(f'\n🛑 target step 도달 ({current_step} >= {TARGET_STOP_STEP}), 학습 프로세스를 중단합니다...')
                stop_requested = True
                with state_lock:
                    runtime_state['stop_requested'] = True

                try:
                    print('[monitor] auto-stop: terminate 전송')
                    proc.terminate()

                    wait_deadline = time.time() + 8
                    while proc.poll() is None and time.time() < wait_deadline:
                        time.sleep(0.2)

                    if proc.poll() is None:
                        print('[warn] terminate timeout -> kill로 전환')
                        proc.kill()

                except Exception as e:
                    print(f'[warn] stop signal 실패: {e}; kill로 전환')
                    try:
                        proc.kill()
                    except Exception as e2:
                        print(f'[warn] kill 실패: {e2}')

        t.join(timeout=5)
        with state_lock:
            suppressed = runtime_state['suppressed_shutdown_lines']

        if stop_requested and suppressed > 0:
            print(f'[monitor] shutdown 노이즈 라인 {suppressed}개를 숨겼습니다 (auto-stop 단계).')

        post_step, post_step_mtime, post_step_source = read_step_snapshot(LOG_DIR, RUN_ID)
        if post_step is not None:
            post_age_txt = 'unknown'
            if post_step_mtime is not None:
                post_age_txt = f'{(time.time() - post_step_mtime):.0f}s'
            print(f'[post-check] snapshot_step={post_step} / source={post_step_source or "n/a"} / age={post_age_txt}')

        if TARGET_STOP_STEP is not None and stop_requested:
            if proc.returncode not in (0, None):
                print(f'\n학습 종료 (exit={proc.returncode}) - target auto-stop에 따른 종료로 처리합니다.')
            else:
                print(f'\n학습 종료 (exit={proc.returncode})')
            print('다음 실행은 RESUME=True로 같은 RUN_ID를 사용하세요.')
        else:
            print(f'\n학습 종료 (exit={proc.returncode})')
        break

    attempt += 1

else:
    raise RuntimeError('mlagents-learn 시작 실패 — 위 출력 내용을 확인하세요.')

Mode: FORCE — Warm-start from: evader_s1_obstacle_44d_v13bguard_seed42
실행 커맨드:
c:\IIT_DroneLearning\.venv\Scripts\mlagents-learn.exe c:\IIT_DroneLearning\python\config\evader_s1_obstacle_20260408_v13c_rootfix.yaml --run-id=evader_s1_obstacle_44d_v13crootfix_seed42 --results-dir=c:\IIT_DroneLearning\python\results --force --initialize-from=evader_s1_obstacle_44d_v13bguard_seed42

Auto-stop target step: 100000
[init] metadata saved: c:\IIT_DroneLearning\python\results\evader_s1_obstacle_44d_v13crootfix_seed42\init_metadata.json
⏳ mlagents-learn 시작 중... (포트 5004 준비 대기)

            ┐  ╖
        ╓╖╬│╡  ││╬╖╖
    ╓╖╬│││││┘  ╬│││││╬╖
 ╖╬│││││╬╜        ╙╬│││││╖╖                               ╗╗╗
 ╬╬╬╬╖││╦╖        ╖╬││╗╣╣╣╬      ╟╣╣╬    ╟╣╣╣             ╜╜╜  ╟╣╣
 ╬╬╬╬╬╬╬╬╖│╬╖╖╓╬╪│╓╣╣╣╣╣╣╣╬      ╟╣╣╬    ╟╣╣╣ ╒╣╣╖╗╣╣╣╗   ╣╣╣ ╣╣╣╣╣╣ ╟╣╣╖   ╣╣╣
 ╬╬╬╬┐  ╙╬╬╬╬│╓╣╣╣╝╜  ╫╣╣╣╬      ╟╣╣╬    ╟╣╣╣ ╟╣╣╣╙ ╙╣╣╣  ╣╣╣ ╙╟╣╣╜╙  ╫╣╣  ╟╣╣
 ╬╬╬╬┐     ╙╬╬╣╣      ╫╣╣╣╬      ╟╣╣╬    ╟╣╣╣ ╟╣╣╬   ╣╣╣  ╣╣╣  ╟╣╣     ╣╣╣┌╣

---
## 6. TensorBoard 모니터링

학습 중 **새 터미널**에서 아래 명령어로 TensorBoard를 실행하세요.


In [122]:
tb_cmd = f'.venv\\Scripts\\tensorboard --logdir python/results/{RUN_ID} --port 6006'
print('새 터미널에서 실행:')
print(tb_cmd)
print()
print('브라우저: http://localhost:6006')
print()
print('주요 모니터링 지표:')
print('  Environment/Cumulative Reward  → 상승 추세 확인 (초반 1~3 기대)')
print('  Environment/Episode Length     → 500~1000 (탐색 중 정상)')
print('  Policy/Entropy                 → 서서히 감소해야 함')
print('  Diagnostics/TermGoal           → goal 종료 비율 (window 평균)')
print('  Diagnostics/TermTimeout        → timeout 종료 비율 (window 평균)')
print('  Diagnostics/TermCrash          → crash 종료 비율 (window 평균)')
print('  Diagnostics/TermCrashOuterWall → 외벽 충돌 종료 비율 (window 평균)')
print('  Diagnostics/TermCrashBuilding  → 건물 충돌 종료 비율 (window 평균)')
print('  Diagnostics/TermCrashBoundaryOverflow → 경계 이탈 종료 비율 (window 평균)')
print('  Diagnostics/TermCrashPenalty   → crash terminal penalty 평균')
print('  Diagnostics/TermCaptured       → captured 종료 비율 (window 평균)')
print('  Diagnostics/TermEpisodeSteps   → 종료 시 평균 step')
print('  Diagnostics/TermGoalDistance   → 종료 시 goal까지 평균 거리')
print('  Diagnostics/StagnationPenalty  → 정체 페널티 발생 빈도')
print('  Diagnostics/WallSafetyActivated      → 벽 초근접 안전 보정 발동 빈도')
print('  Diagnostics/WallSafetyForwardBlocked → 벽 전방에서 forward pitch 차단 빈도')
print('  Diagnostics/WallSafetyFrontRisk      → 벽 전방 위험도 평균 (0~1)')
print('  Diagnostics/WallSafetyAvoidBias      → 좌우 회피 바이어스 평균 (-1~1)')
print('  Diagnostics/WallSafetyReverseAssist  → 제한 역피치 보정 강도 평균 (0~1)')
print('  Diagnostics/BoundarySafetyActivated  → 경계 복귀 보정 발동 빈도')
print('  Diagnostics/BoundaryRisk             → 위치 기반 경계 위험도 평균')
print('  Diagnostics/BoundaryEffectiveRisk    → 속도 반영 최종 경계 위험도 평균')
print('  Diagnostics/BoundaryOutwardSpeed     → 경계 외향 속도 평균')
print('  Diagnostics/BoundaryRiskPenalty      → 경계 위험 페널티 강도')
print('  Diagnostics/GoalClearanceFailureType → goal 배치 실패 원인 코드')
print('  Diagnostics/GoalClearanceFailBoundary -> goal 경계 근접 실패 빈도')
print()
print('케이스 확인 포인트:')
print('  - goal 망각/정체: TermTimeout↑ + TermGoalDistance↑ + StagnationPenalty↑')
print('  - 충돌 과다: TermCrash↑ + EpisodeLength↓')
print('  - 벽 회피 미작동: TermCrash↑ 인데 WallSafetyActivated≈0')
print('  - 벽 위험 고착: WallSafetyFrontRisk↑ + WallSafetyForwardBlocked↑ + TermCrashBuilding↑')
print('  - 경계 과속 이탈: BoundaryOutwardSpeed↑ + BoundaryEffectiveRisk↑ + TermCrashBoundaryOverflow↑')
print('  - 재개 품질: resume 후 10k~30k 구간에서 Cumulative Reward 급락 여부')
print()
print('⚠️  위험 신호:')
print('  - 50k 스텝 이후에도 Mean Reward 음수 지속 → 보상 재설계 필요')
print('  - TermTimeout 높고 TermCrash 낮음 + GoalDistance 큼 → goal 망각/정체 가능성')
print('  - TermCrash 높고 Episode Length 짧음 → 충돌/기동 안정성 이슈')
print('  - WallSafetyActivated 낮은데 TermCrash 높음 → 센서/DetectionLayerMask 설정 점검 필요')
print('  - WallSafetyReverseAssist 높고 진동 증가 → reverse assist 상한/스무딩 점검 필요')


새 터미널에서 실행:
.venv\Scripts\tensorboard --logdir python/results/evader_s1_obstacle_44d_v13crootfix_seed42 --port 6006

브라우저: http://localhost:6006

주요 모니터링 지표:
  Environment/Cumulative Reward  → 상승 추세 확인 (초반 1~3 기대)
  Environment/Episode Length     → 500~1000 (탐색 중 정상)
  Policy/Entropy                 → 서서히 감소해야 함
  Diagnostics/TermGoal           → goal 종료 비율 (window 평균)
  Diagnostics/TermTimeout        → timeout 종료 비율 (window 평균)
  Diagnostics/TermCrash          → crash 종료 비율 (window 평균)
  Diagnostics/TermCrashOuterWall → 외벽 충돌 종료 비율 (window 평균)
  Diagnostics/TermCrashBuilding  → 건물 충돌 종료 비율 (window 평균)
  Diagnostics/TermCrashBoundaryOverflow → 경계 이탈 종료 비율 (window 평균)
  Diagnostics/TermCrashPenalty   → crash terminal penalty 평균
  Diagnostics/TermCaptured       → captured 종료 비율 (window 평균)
  Diagnostics/TermEpisodeSteps   → 종료 시 평균 step
  Diagnostics/TermGoalDistance   → 종료 시 goal까지 평균 거리
  Diagnostics/StagnationPenalty  → 정체 페널티 발생 빈도
  Diagnostics/WallSafetyActivated      → 벽 초근접 안전 보정

### 6-1. 케이스 진단 스냅샷 (TensorBoard 이벤트 직접 확인)

아래 셀은 최근 이벤트 파일에서 핵심 진단 지표의 최신값과 최근 평균을 읽어,
앞서 정의한 edge case(정체/충돌/벽 회피 미작동)를 빠르게 점검합니다.

In [123]:
from pathlib import Path
from statistics import mean

try:
    from tensorboard.backend.event_processing import event_accumulator
except Exception as e:
    print(f'[diag] tensorboard event_accumulator import 실패: {e}')
    print('[diag] .venv 환경에서 tensorboard 설치 상태를 확인하세요.')
    raise

run_tb_dir = LOG_DIR / RUN_ID / 'Drone_Evader'
event_files = sorted(run_tb_dir.glob('events.out.tfevents.*'))
if not event_files:
    print(f'[diag] 이벤트 파일이 없습니다: {run_tb_dir}')
else:
    latest_event = event_files[-1]
    print(f'[diag] latest event file: {latest_event.name}')

    ea = event_accumulator.EventAccumulator(
        str(latest_event),
        size_guidance={event_accumulator.SCALARS: 0}
    )
    ea.Reload()
    scalar_tags = set(ea.Tags().get('scalars', []))

    wanted = [
        'Environment/Cumulative Reward',
        'Environment/Episode Length',
        'Diagnostics/TermGoal',
        'Diagnostics/TermTimeout',
        'Diagnostics/TermCrash',
        'Diagnostics/TermCrashOuterWall',
        'Diagnostics/TermCrashBuilding',
        'Diagnostics/TermCrashBoundaryOverflow',
        'Diagnostics/TermCrashOther',
        'Diagnostics/TermCrashPenalty',
        'Diagnostics/TermCaptured',
        'Diagnostics/TermGoalDistance',
        'Diagnostics/StagnationPenalty',
        'Diagnostics/WallSafetyActivated',
        'Diagnostics/WallSafetyForwardBlocked',
        'Diagnostics/WallSafetyFrontRisk',
        'Diagnostics/WallSafetyAvoidBias',
        'Diagnostics/WallSafetyReverseAssist',
        'Diagnostics/BoundarySafetyActivated',
        'Diagnostics/BoundaryRisk',
        'Diagnostics/BoundaryEffectiveRisk',
        'Diagnostics/BoundaryOutwardSpeed',
        'Diagnostics/BoundaryRiskPenalty',
        'Diagnostics/GoalClearanceFailureType',
        'Diagnostics/GoalClearanceFailBoundary',
    ]

    print('[diag] scalar summary (latest / mean(last20)):')
    summary = {}
    for tag in wanted:
        if tag not in scalar_tags:
            print(f'  - {tag}: (missing)')
            continue

        points = ea.Scalars(tag)
        values = [p.value for p in points]
        latest = values[-1]
        recent = values[-20:] if len(values) >= 20 else values
        recent_mean = mean(recent) if recent else float('nan')
        summary[tag] = {'latest': latest, 'mean20': recent_mean, 'n': len(values)}
        print(f'  - {tag}: latest={latest:.4f}, mean20={recent_mean:.4f}, n={len(values)}')

    print()
    print('[diag] quick case checks:')

    term_timeout = summary.get('Diagnostics/TermTimeout', {}).get('mean20')
    term_crash = summary.get('Diagnostics/TermCrash', {}).get('mean20')
    term_crash_outer = summary.get('Diagnostics/TermCrashOuterWall', {}).get('mean20')
    term_crash_building = summary.get('Diagnostics/TermCrashBuilding', {}).get('mean20')
    term_crash_boundary = summary.get('Diagnostics/TermCrashBoundaryOverflow', {}).get('mean20')
    term_crash_penalty = summary.get('Diagnostics/TermCrashPenalty', {}).get('mean20')
    goal_dist = summary.get('Diagnostics/TermGoalDistance', {}).get('mean20')
    stag_pen = summary.get('Diagnostics/StagnationPenalty', {}).get('mean20')
    wall_safe = summary.get('Diagnostics/WallSafetyActivated', {}).get('mean20')
    wall_front_risk = summary.get('Diagnostics/WallSafetyFrontRisk', {}).get('mean20')
    wall_avoid_bias = summary.get('Diagnostics/WallSafetyAvoidBias', {}).get('mean20')
    wall_reverse_assist = summary.get('Diagnostics/WallSafetyReverseAssist', {}).get('mean20')
    boundary_safe = summary.get('Diagnostics/BoundarySafetyActivated', {}).get('mean20')
    boundary_risk = summary.get('Diagnostics/BoundaryRisk', {}).get('mean20')
    boundary_effective_risk = summary.get('Diagnostics/BoundaryEffectiveRisk', {}).get('mean20')
    boundary_outward_speed = summary.get('Diagnostics/BoundaryOutwardSpeed', {}).get('mean20')
    boundary_penalty = summary.get('Diagnostics/BoundaryRiskPenalty', {}).get('mean20')
    goal_clear_boundary = summary.get('Diagnostics/GoalClearanceFailBoundary', {}).get('mean20')

    if term_timeout is not None and goal_dist is not None and stag_pen is not None:
        if goal_dist > 50 and stag_pen > 0.5:
            print('  - [warn] goalDistance/stagnation이 매우 높습니다. 장거리 정체 또는 비효율 경로 가능성이 큽니다.')
        elif term_timeout > 0.45 and goal_dist > 10 and stag_pen > 0.05:
            print('  - [warn] goal 망각/정체 시그널이 큽니다 (timeout↑, goalDistance↑, stagnation↑).')
        elif term_timeout > 0.30 and goal_dist > 30:
            print('  - [warn] timeout과 goalDistance가 함께 높아 우회 실패/정체 가능성이 있습니다.')
        else:
            print('  - [ok] goal 망각/정체 시그널이 임계치를 크게 넘지 않습니다.')
    else:
        print('  - [skip] goal 망각/정체 판정용 지표가 아직 부족합니다.')

    if term_crash is not None:
        if term_crash > 0.35:
            print('  - [warn] crash 비율이 높습니다. 벽 근접 회피와 입력 완화 파라미터 재점검이 필요합니다.')
        else:
            print('  - [ok] crash 비율이 치명 임계치를 크게 넘지 않습니다.')
    else:
        print('  - [skip] crash 판정용 지표가 아직 부족합니다.')

    if term_crash_outer is not None or term_crash_building is not None or term_crash_boundary is not None:
        outer_txt = f"{term_crash_outer:.3f}" if term_crash_outer is not None else 'n/a'
        building_txt = f"{term_crash_building:.3f}" if term_crash_building is not None else 'n/a'
        boundary_txt = f"{term_crash_boundary:.3f}" if term_crash_boundary is not None else 'n/a'
        penalty_txt = f"{term_crash_penalty:.3f}" if term_crash_penalty is not None else 'n/a'
        print(f'  - [diag] crash subtype mean20 outer={outer_txt}, building={building_txt}, boundary={boundary_txt}, penalty={penalty_txt}')

        if term_crash_outer is not None and term_crash_outer > 0.20:
            print('  - [warn] outer wall crash 비중이 높습니다. WallSafety 및 경계 근접 행동을 우선 점검하세요.')
        if term_crash_building is not None and term_crash_building > 0.20:
            print('  - [warn] building crash 비중이 높습니다. Goal clearance 반경/프로브를 우선 점검하세요.')
        if term_crash_building is not None and term_crash_building > 0.20 and wall_front_risk is not None and wall_front_risk < 0.10:
            print('  - [warn] building crash 대비 WallSafetyFrontRisk 반응이 낮습니다. 벽 감지 거리/리스크 스케일을 점검하세요.')
        if term_crash_boundary is not None and term_crash_boundary > 0.10:
            print('  - [warn] boundary overflow가 높습니다. 경계 근접 억제 파라미터를 우선 점검하세요.')
    else:
        print('  - [skip] crash subtype 지표가 아직 부족합니다. (코드 업데이트 직후 missing 가능)')

    if wall_front_risk is not None or wall_avoid_bias is not None or wall_reverse_assist is not None:
        wall_risk_txt = f"{wall_front_risk:.3f}" if wall_front_risk is not None else 'n/a'
        wall_bias_txt = f"{wall_avoid_bias:.3f}" if wall_avoid_bias is not None else 'n/a'
        wall_rev_txt = f"{wall_reverse_assist:.3f}" if wall_reverse_assist is not None else 'n/a'
        print(f'  - [diag] wall safety mean20 frontRisk={wall_risk_txt}, avoidBias={wall_bias_txt}, reverseAssist={wall_rev_txt}')

        if term_crash is not None and term_crash > 0.35 and wall_reverse_assist is not None and wall_reverse_assist > 0.20:
            print('  - [warn] crash 구간에서 reverseAssist가 높습니다. 역피치 상한/완화 값을 보수적으로 조정하세요.')
        if term_crash is not None and term_crash > 0.35 and wall_avoid_bias is not None and abs(wall_avoid_bias) < 0.01:
            print('  - [warn] crash 구간에서 avoidBias 반응이 거의 없습니다. 좌우 회피 유도 강도/스무딩을 점검하세요.')
    else:
        print('  - [skip] wall safety 상세 지표가 아직 부족합니다. (코드 업데이트 직후 missing 가능)')

    if boundary_safe is not None or boundary_risk is not None or boundary_penalty is not None:
        bsafe_txt = f"{boundary_safe:.3f}" if boundary_safe is not None else 'n/a'
        brisk_txt = f"{boundary_risk:.3f}" if boundary_risk is not None else 'n/a'
        berisk_txt = f"{boundary_effective_risk:.3f}" if boundary_effective_risk is not None else 'n/a'
        bout_txt = f"{boundary_outward_speed:.3f}" if boundary_outward_speed is not None else 'n/a'
        bpen_txt = f"{boundary_penalty:.4f}" if boundary_penalty is not None else 'n/a'
        print(f'  - [diag] boundary safety mean20 activated={bsafe_txt}, risk={brisk_txt}, effectiveRisk={berisk_txt}, outwardSpeed={bout_txt}, penalty={bpen_txt}')

        if term_crash_boundary is not None and term_crash_boundary > 0.10 and boundary_safe is not None and boundary_safe < 0.20:
            print('  - [warn] 경계 crash 대비 BoundarySafety 발동이 낮습니다. margin/assist를 상향하세요.')
        if term_crash_boundary is not None and term_crash_boundary > 0.10 and boundary_effective_risk is not None and boundary_effective_risk < 0.10:
            print('  - [warn] boundary crash 대비 effective risk 반응이 낮습니다. outward speed reference를 낮추세요.')
    else:
        print('  - [skip] boundary safety 판정용 지표가 아직 부족합니다.')

    if goal_clear_boundary is not None and goal_clear_boundary > 0.05:
        print('  - [warn] goal 경계 근접 재샘플링 실패가 반복됩니다. goal placement margin/attempt를 재점검하세요.')

    if term_crash is not None and wall_safe is not None:
        if term_crash > 0.35 and wall_safe < 0.01 and (wall_front_risk is None or wall_front_risk < 0.05):
            print('  - [warn] crash 대비 WallSafety 발동/위험 인지가 모두 낮습니다. 센서 마스크/레이 설정 점검이 필요합니다.')
        else:
            print('  - [ok] crash 대비 WallSafety 발동 관계가 비정상으로 보이지 않습니다.')
    else:
        print('  - [skip] wall safety 판정용 지표가 아직 부족합니다. (코드 업데이트 직후에는 missing일 수 있음)')

[diag] latest event file: events.out.tfevents.1775651326.DESKTOP-A8AHOBI.38436.0
[diag] scalar summary (latest / mean(last20)):
  - Environment/Cumulative Reward: latest=10.6075, mean20=8.4809, n=20
  - Environment/Episode Length: latest=104.1489, mean20=122.1735, n=20
  - Diagnostics/TermGoal: latest=0.5312, mean20=0.4725, n=20
  - Diagnostics/TermTimeout: latest=0.0000, mean20=0.0079, n=20
  - Diagnostics/TermCrash: latest=0.4688, mean20=0.5197, n=20
  - Diagnostics/TermCrashOuterWall: latest=0.0000, mean20=0.0000, n=20
  - Diagnostics/TermCrashBuilding: latest=0.4219, mean20=0.3836, n=20
  - Diagnostics/TermCrashBoundaryOverflow: latest=0.0469, mean20=0.1361, n=20
  - Diagnostics/TermCrashOther: latest=0.0000, mean20=0.0000, n=20
  - Diagnostics/TermCrashPenalty: latest=-0.7336, mean20=-0.7839, n=20
  - Diagnostics/TermCaptured: latest=0.0000, mean20=0.0000, n=20
  - Diagnostics/TermGoalDistance: latest=50.5916, mean20=54.3273, n=20
  - Diagnostics/StagnationPenalty: latest=1.0000, 

---
## 7. 결과 확인 및 ONNX 경로

In [112]:
run_dir = LOG_DIR / RUN_ID

if run_dir.exists():
    onnx_files = list(run_dir.glob('**/*.onnx'))
    pt_files   = list(run_dir.glob('**/*.pt'))

    print(f'✅ 결과 폴더: {run_dir}')
    print(f'ONNX 파일 ({len(onnx_files)}개):')
    for f in sorted(onnx_files):
        print(f'  {f.name}')
    print(f'체크포인트 ({len(pt_files)}개): {len(pt_files)}개')
else:
    print(f'❌ 결과 폴더 없음: {run_dir}')
    print('5번 셀(학습)을 먼저 실행하세요.')


✅ 결과 폴더: c:\IIT_DroneLearning\python\results\evader_s1_obstacle_44d_v13bguard_seed42
ONNX 파일 (4개):
  Drone_Evader-24976.onnx
  Drone_Evader-49925.onnx
  Drone_Evader-74996.onnx
  Drone_Evader-99906.onnx
체크포인트 (5개): 5개


---
## 8. Eval 실행 (고정 시드)

1. Unity에서 평가 모드로 `EVAL_RUN_ID` 모델을 로드해 에피소드 로그(`episodes.csv` 또는 `episodes.ndjson`)를 생성합니다.
2. 아래 셀을 실행해 지표를 계산하고 `metrics.json`을 저장합니다.
3. 결과를 `docs/EXPERIMENTS.md`에 기록합니다.

In [113]:
import subprocess, sys

run_dir = REPO_PATH / "python" / "results" / EVAL_RUN_ID
csv_log = run_dir / "episodes.csv"
ndjson_log = run_dir / "episodes.ndjson"

if not csv_log.exists() and not ndjson_log.exists():
    print("[eval] episode log가 없어 평가를 실행할 수 없습니다.")
    print(f"  - expected: {csv_log}")
    print(f"  - expected: {ndjson_log}")
    print("  - Unity에서 Eval 모드로 모델을 실행해 episodes.csv 또는 episodes.ndjson를 먼저 export 하세요.")
else:
    eval_script = REPO_PATH / "python" / "scripts" / "eval.py"
    assert eval_script.exists(), f"eval.py 없음: {eval_script}"

    eval_cmd = [
        sys.executable,
        str(eval_script),
        "--run-id", EVAL_RUN_ID,
        "--n-episodes", str(EVAL_EPISODES),
        "--seed", str(EVAL_SEED),
    ]

    print("Eval command:")
    print(" ".join(eval_cmd))
    print()

    result = subprocess.run(eval_cmd, cwd=str(REPO_PATH), text=True)
    if result.returncode != 0:
        print()
        print("[hint] episodes.csv/episodes.ndjson가 없으면 Unity 평가 실행 후 로그를 먼저 export 하세요.")

[eval] episode log가 없어 평가를 실행할 수 없습니다.
  - expected: c:\IIT_DroneLearning\python\results\evader_s1_obstacle_44d_v13bguard_seed42\episodes.csv
  - expected: c:\IIT_DroneLearning\python\results\evader_s1_obstacle_44d_v13bguard_seed42\episodes.ndjson
  - Unity에서 Eval 모드로 모델을 실행해 episodes.csv 또는 episodes.ndjson를 먼저 export 하세요.


---
## 9. Stage1-B 전환 (수렴 확인 후)

수렴 기준:
- `goal_reach_rate ≥ 30%`
- `crash_rate ≤ 15%`
- 최근 100k 구간에서 mean reward 급락 없음

**Unity Inspector 변경 (Stage1-B):**
- `EvaderAgent._goalOnlyMode = false` (RL Pursuer 활성)
- Pursuer 오브젝트 -> BehaviorParameters -> Model = `pursuer_s2_catch_v3_499980.onnx`

**다음 버전 설정 예시:**
```python
# Stage1-B는 Stage1-A 최종 run을 초기값으로 사용
INIT_FROM = f"evader_s1_obstacle_44d_v7_seed{SEED}"  # 또는 best obstacle run-id
RUN_ID    = f"evader_s1_pursuer_44d_v1_seed{SEED}"
RESUME    = False
TARGET_STOP_STEP = 300000
```
